Part 1: The Conditional Workflow

In this workflow, a router function acts as a decision-maker. It checks the state and dynamically determines whether to send the task to the researcher or skips straight to the executor.

In [7]:
# 1. Define the Shared State
state = {
    "query": "Sweden Leave Policy",
    "needs_research": True,  # This flag will dictate the path
    "plan": "",
    "documents": [],
    "answer": ""
}

# 2. Define the Agent Nodes
def planner(s):
    print("[Planner] Creating plan...")
    s["plan"] = "Determine if research is needed."
    return s

def researcher(s):
    print("[Researcher] Searching databases...")
    s["documents"] = ["Sweden Leave Policy 2026 - Draft v2"]
    return s 

def executor(s):
    print("[Executor] Writing final answer...")
    if s["documents"]:
        s["answer"] = f"Based on research, found: {s['documents'][0]}"
    else:
        s["answer"] = "Answered using general knowledge."
    return s 

# 3. Define the Router (The Conditional Switch)
def router(s):
    if s["needs_research"]:
        print("[Router] Path chosen: Research is required.")
        return "researcher"
    else:
        print("[Router] Path chosen: Skipping research.")
        return "executor"

# 4. Execute the Conditional Pipeline
# Step 1: Always run the planner first
state = planner(state)

# Step 2: Make a decision using the router
next_step = router(state)

# Step 3: Follow the decided path
if next_step == "researcher":
    state = researcher(state)
    state = executor(state)
elif next_step == "executor":
    state = executor(state)

print("\nFinal Answer:", state["answer"])


[Planner] Creating plan...
[Router] Path chosen: Research is required.
[Researcher] Searching databases...
[Executor] Writing final answer...

Final Answer: Based on research, found: Sweden Leave Policy 2026 - Draft v2


In [8]:
%pip install -q google-genai

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from google import genai

# Initialize the Gemini client (Make sure GEMINI_API_KEY is set in your environment variables)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

# 1. Define the Shared State
state = {
    "query": "What is the mandatory minimum annual leave in Sweden?",
    "plan": "",
    "documents": [],
    "answer": ""
}

# 2. Define LLM-powered Agents
def llm_planner(s):
    print("🧠 [LLM Planner] Analyzing query...")
    prompt = f"Create a short execution plan to answer this user query: '{s['query']}'"
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    s["plan"] = response.text.strip()
    return s

def llm_researcher(s):
    print("🧠 [LLM Researcher] Gathering facts...")
    prompt = f"Act as a researcher. Provide 2-3 key legal facts about this topic: '{s['query']}'"
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    # Store the generated facts as our document source
    s["documents"] = [response.text.strip()]
    return s 

def llm_executor(s):
    print("🧠 [LLM Executor] Synthesizing final response...")
    prompt = f"""
    User Query: {s['query']}
    Execution Plan: {s['plan']}
    Research Gathered: {s['documents'][0]}
    
    Synthesize the research into a concise, professional answer for the user.
    """
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )
    s["answer"] = response.text.strip()
    return s 

# 3. Run the Sequential LLM Loop
for agent in (llm_planner, llm_researcher, llm_executor):
    state = agent(state)

# 4. View Results
print("\n--- Final Workflow Results ---")
print(f"Plan Generated:\n{state['plan']}\n")
print(f"Final Answer:\n{state['answer']}")


ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.